<a href="https://colab.research.google.com/github/Ghan539/Gen_AI-Workshop/blob/main/GenAI_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import tensorflow as tf
from tensorflow.keras import Sequential, layers
import matplotlib.pyplot as plt

# Load and normalize MNIST
(x, _), _ = tf.keras.datasets.mnist.load_data()
x = x.astype("float32").reshape(-1, 784) / 127.5 - 1

# Generator: Noise -> Image
G = Sequential([
    layers.Input(shape=(100,)),
    layers.Dense(256, activation="relu"),
    layers.Dense(784, activation="tanh")
])

# Discriminator: Image -> Real/Fake
D = Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

loss = tf.keras.losses.BinaryCrossentropy()
g_opt = tf.keras.optimizers.Adam(0.0002)
d_opt = tf.keras.optimizers.Adam(0.0002)

# Training
for epoch in range(20):
    for real in tf.data.Dataset.from_tensor_slices(x).shuffle(60000).batch(128):
        n = tf.shape(real)[0]
        noise = tf.random.normal((n, 100))

        # Train Discriminator
        with tf.GradientTape() as tape:
            fake = G(noise)
            d_loss = loss(tf.ones((n, 1)), D(real)) + \
                     loss(tf.zeros((n, 1)), D(fake))
        grads = tape.gradient(d_loss, D.trainable_variables)
        d_opt.apply_gradients(zip(grads, D.trainable_variables))

        # Train Generator
        with tf.GradientTape() as tape:
            fake = G(tf.random.normal((n, 100)))
            g_loss = loss(tf.ones((n, 1)), D(fake))
        grads = tape.gradient(g_loss, G.trainable_variables)
        g_opt.apply_gradients(zip(grads, G.trainable_variables))

    print(f"Epoch {epoch+1}: D={d_loss:.3f}, G={g_loss:.3f}")

# Generate 10 synthetic digits
images = G(tf.random.normal((10, 100))).numpy()

plt.figure(figsize=(12, 2))
for i in range(10):
    plt.subplot(1, 10, i+1)
    plt.imshow(images[i].reshape(28, 28), cmap="gray",
               vmin=-1, vmax=1)
    plt.axis("off")
plt.show()













Epoch 1: D=0.084, G=3.280
Epoch 2: D=0.507, G=1.518
Epoch 3: D=0.366, G=1.840
Epoch 4: D=0.550, G=1.845
Epoch 5: D=0.523, G=1.972
Epoch 6: D=0.438, G=2.401
Epoch 7: D=0.388, G=2.189
Epoch 8: D=0.580, G=1.559
Epoch 9: D=0.475, G=1.987
Epoch 10: D=0.509, G=1.931
Epoch 11: D=0.283, G=2.816
Epoch 12: D=0.573, G=1.719
Epoch 13: D=0.393, G=2.341
Epoch 14: D=0.393, G=2.379
Epoch 15: D=0.313, G=3.016
